# M31 · Locating an earthquake

**Part 2 · week 4 · about 11 h core + 1 h stretch**

## 한국어 요약

이 모듈에서는 지진파의 도달 시각으로 진원(위치와 깊이)과 발생 시각을 함께 구하는 절대 진원 결정을 직접 구현합니다. 먼저 정답을 아는 가상의 지진으로 격자 탐색과 가이거 방법(야코비 행렬과 감쇠 최소제곱)을 작성해 검증하고, 공분산 행렬로 위치의 표준 오차와 깊이-발생 시각 사이의 상관을 계산한 뒤 몬테카를로 실험으로 확인합니다. 다음으로 M20에서 직접 고른 피킹과 M30에서 만든 주시 함수(kim1983 모델의 최초 도달파)로 2019년 상주 지진(ML 3.9)의 진원을 결정하고 기상청 목록과 비교합니다. 2018년 상주의 작은 지진(ML 1.4)은 직접 피킹한 뒤, 가장 가까운 관측소가 있을 때와 없을 때 깊이가 얼마나 잘 정해지는지를 깊이 탐색과 표준 오차로 측정합니다. 마지막으로 실무 진원 결정 프로그램 HYPOINVERSE를 같은 피킹으로 실행하고(변경 없는 재실행, 그다음 변수 하나 변경) 그 결과를 여러분의 위치와 비교해 설명합니다.

## What you will be able to do

- Locate a made-up earthquake by grid search and by Geiger's method (damped least squares with the Jacobian of the travel times), and test both against a hypocenter known by construction
- Compute the covariance matrix, the standard errors and the correlation of depth and origin time of a location, and confirm them with a Monte Carlo test of your own locator
- Locate the 2019 Sangju mainshock from your own picks with the first arrivals of kim1983 computed by your M30 functions, and compare the result with the KMA catalog
- Measure with a depth scan and the standard errors how well the depth of the 2018 Sangju earthquake is known with and without the station nearest to it
- Run HYPOINVERSE as a production locator on your picks (an unchanged rerun, then one changed setting) and explain how its locations relate to yours

## Before you start

- Modules: M20, M30
- Reading: Havskov, Bormann & Schweitzer (2012), sections 1 to 4 (the location problem, the iterative solution, errors); Shearer (2019), the chapter on earthquake location; Husen & Hardebeck (2010), the whole article (what makes a location accurate); Lomax, Michelini & Curtis (2009), the introduction and the section on grid search
- Needs: stp, pocketquake
- Key terms (see the glossary): absolute location (절대 진원 결정), hypocenter (진원), epicenter (진앙), origin time (진원시, 발생 시각), focal depth (진원 깊이), phase pick (위상 피킹, 도달 시각 읽기), arrival time (도달 시각), travel time (주시), residual (잔차), RMS residual (RMS 잔차), misfit function (목적 함수, 불일치 함수), local coordinates (국지 좌표계), map projection (지도 투영법), epicentral distance (진앙 거리), half-space (반무한 공간), grid search (격자 탐색), Geiger's method (가이거 방법), linearization (선형화), Jacobian matrix (야코비 행렬), finite difference (유한 차분), least squares (최소제곱법), normal equations (정규 방정식), damping (감쇠 정규화, 댐핑), stopping rule (종료 조건 (수렴 판정)), airquake (공중 지진 (에어퀘이크)), covariance matrix (공분산 행렬), standard error (표준 오차), error correlation (오차 상관 (상관 계수)), parameter trade-off (매개변수 상충 관계), error ellipse (오차 타원), Monte Carlo simulation (몬테카를로 시뮬레이션), first arrival (최초 도달파 (초동)), head wave (선두파), Pn and Sn (Pn파와 Sn파), crossover distance (교차 거리), travel-time table (주시표), azimuthal gap (방위각 공백), depth scan (깊이 탐색 (고정 깊이 탐색)), HYPOINVERSE (진원 결정 프로그램), phase file (위상 파일 (도달 시각 파일)), fixed-width format (고정 폭 형식), implied decimal point (생략된 소수점), weight code (가중치 코드), summary file (HYPOINVERSE; 요약 파일 (HYPOINVERSE)), trial depth (시작 깊이), distance weighting (거리 가중), residual weighting (잔차 가중), deterministic computation (결정론적 계산), Wadati diagram (와다티 도표), Vp/Vs ratio (Vp/Vs 비), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M31.yaml](../../../.sgtl/criteria/M31.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M31.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
SMALL_EVENT_ID = "2018000078"             # the ML 1.4 Sangju earthquake of 25 February 2018 (KST) in the KMA catalog
PICKS_CSV = "../picks/M20_manual_picks.csv"           # your manual picks of module M20 (in work/picks/)
SMALL_PICKS_CSV = "../picks/M31_2018_manual_picks.csv"  # your picks of the 2018 earthquake (Part 6 writes it)
MODEL_NAME = "kim1983"                    # the velocity model of Parts 5 to 7
MAX_DISTANCE_KM = 100.0                   # picks farther than this from the KMA epicenter are not used
GRID_HALF_KM = 20.0                       # grid search: +- this many km east and north of the KMA epicenter
GRID_STEP_KM = 1.0                        # grid search: node spacing (km)
GRID_DEPTHS_KM = (0.5, 30.0, 1.0)         # grid search: first depth, last depth, step (km)
SCAN_DEPTHS_KM = (0.5, 25.0, 0.5)         # task 5, depth scan: first depth, last depth, step (km)
SMALL_BAND = "HH"                         # Part 6: the band you pick the 2018 earthquake on
SMALL_MAX_KM = 60.0                       # Part 6: the picker lists stations within this epicentral distance
TRIAL_DEPTH_KM = 10.0                     # task 6: HYPOINVERSE's trial depth in the lab's settings
CHANGED_TRIAL_DEPTH_KM = 5.0              # task 6: the one setting you change (HYPOINVERSE's trial depth)
PRACTICE_SEED = 31                        # seed of the made-up practice earthquake (Part 1)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M31")

import filecmp
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read
from pyproj import Proj
from scipy.interpolate import RectBivariateSpline

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M31-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up earthquakes whose hypocenters are known
from how they were built (`st.reroll("M31-T1")` draws new ones), compare them with independent programs (SciPy's
least-squares solvers, which your own functions may not use, and HYPOINVERSE), and, in task 4, test your location
of the Sangju mainshock against your own picks and loose limits only, never printing a value.

The order of the work is the one of every module from here on: first you write the core functions and test them
on a made-up earthquake with a known answer (tasks 1-3), then you apply them to real data and compare with an
independent reference (tasks 4 and 5), and last you run the production tool, an unchanged rerun first, then with
one setting changed (task 6).

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M31.md`.

## Part 1 · Four unknowns from arrival times (about 25 min)

**Absolute location** (glossary `absolute_location`) finds one earthquake's **hypocenter** (glossary
`hypocenter`), the point where the rupture starts, and its **origin time** (glossary `origin_time`) from the
**arrival times** (glossary `arrival_time`) of its waves at the stations, the **phase picks** (glossary
`phase_pick`) of M20. The hypocenter has three coordinates: east, north and the **focal depth** (glossary
`focal_depth`); the **epicenter** (glossary `epicenter`) is the point on the surface above it. With the origin
time that makes four unknowns: $x, y, z, t_0$.

A pick $i$ of phase P or S at a station tells

$$t_i = t_0 + T_i(x, y, z) + e_i ,$$

where $T_i$ is the **travel time** (glossary `travel_time`) from the hypocenter to that station for that phase,
which a velocity model predicts (M30), and $e_i$ is the error of the pick and of the model. For a trial
hypocenter and origin time, the **residual** (glossary `residual`) of a pick is $r_i = t_i - t_0 - T_i$, and the
**RMS residual** (glossary `rms_residual`) sums up all of them in one number,

$$\mathrm{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} r_i^2}\,.$$

Locating means finding the $x, y, z, t_0$ that make this **misfit function** (glossary `misfit_function`)
smallest: the RMS is the misfit of this module. With four unknowns you need at least four picks, and many more
to see how good the answer is.

**A local frame.** Over a hundred kilometres the Earth's curvature hardly matters for distances, so we work in
**local coordinates** (glossary `local_coordinates`): $x$ east and $y$ north of a reference point, in km, and
$z$ down from the surface. A **map projection** (glossary `map_projection`) converts latitude and longitude into
$x, y$: we use an azimuthal equidistant projection centered on the reference point (Part 5), which keeps the
distance to the center exact. All stations sit at $z = 0$; their elevations (a few hundred metres at most here)
would change travel times by a few hundredths of a second, which this module leaves out.

**The travel-time function.** Your locators will not know any velocity model. They receive a function
`tt(distance_km, depth_km, phase)` that returns the travel times for a 1-D numpy array of **epicentral
distances** (glossary `epicentral_distance`) from a source at one depth, for phase `"P"` or `"S"`. In Parts 1-4
that function is the simplest model there is, a **half-space** (glossary `half_space`) with straight rays; from
Part 5 on it will be built from your own M30 functions. A locator written this way works with any model.

The practice earthquake below is made up: ten stations with invented names around a hypocenter we chose, and
picks with random errors of 0.05 s, the size of good manual picks. Its true hypocenter is printed, because the
point of the practice is to see whether your locators find it.

In [ ]:
V_HALF = {"P": 6.0, "S": 6.0 / 1.73}         # km/s: P and S speed of the practice half-space


def tt_half(distance_km, depth_km, phase):
    """Travel time (s) in a homogeneous half-space: straight rays at V_HALF[phase] km/s."""
    return np.sqrt(np.asarray(distance_km, dtype=float) ** 2 + float(depth_km) ** 2) / V_HALF[phase]


practice_rng = np.random.default_rng(PRACTICE_SEED)
PRACTICE_TRUTH = {"x_km": 3.4, "y_km": -1.8, "z_km": 8.6, "t0_s": 0.7}
angles = np.sort(practice_rng.uniform(0.0, 2.0 * np.pi, 10))
ranges = practice_rng.uniform(6.0, 45.0, 10)
practice_stations = pd.DataFrame({"station": [f"XX.P{k:02d}" for k in range(1, 11)],
                                  "x_km": np.round(ranges * np.sin(angles), 2),
                                  "y_km": np.round(ranges * np.cos(angles), 2)})
rows = []
for _, sta in practice_stations.iterrows():
    for phase in ("P", "S"):
        if phase == "S" and sta["station"] in ("XX.P03", "XX.P08"):
            continue                                  # two stations without an S pick
        dist = np.hypot(sta["x_km"] - PRACTICE_TRUTH["x_km"], sta["y_km"] - PRACTICE_TRUTH["y_km"])
        travel = tt_half(np.array([dist]), PRACTICE_TRUTH["z_km"], phase)[0]
        rows.append({"station": sta["station"], "phase": phase, "x_km": sta["x_km"], "y_km": sta["y_km"],
                     "t_s": round(PRACTICE_TRUTH["t0_s"] + travel + practice_rng.normal(0.0, 0.05), 3)})
practice = pd.DataFrame(rows)
print("practice earthquake (made up): hypocenter", PRACTICE_TRUTH)
print(practice.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(practice_stations["x_km"], practice_stations["y_km"], "^", color="tab:blue", ms=8, label="Stations")
for _, sta in practice_stations.iterrows():
    ax.annotate(sta["station"][3:], (sta["x_km"], sta["y_km"]), xytext=(4, 3), textcoords="offset points",
                fontsize=8)
ax.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["y_km"], "*", color="gold", mec="k", ms=14, label="True epicenter")
ax.set_xlabel("East (km)")
ax.set_ylabel("North (km)")
ax.set_aspect("equal")
ax.set_title("Practice network", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_network.png", dpi=150)

## Part 2 · Grid search (about 20 min, then task 1)

The plainest way to find the smallest misfit is to try many places. A **grid search** (glossary `grid_search`)
lays a regular grid of trial hypocenters (the **nodes**) over the region and depth range where the earthquake
can be, computes the misfit at every node, and keeps the best one. It needs no derivatives and no starting
point, it cannot get stuck in a dip of the misfit that is not the deepest, and the misfit at every node is a
picture of how well the data fix the hypocenter (Lomax et al. 2009). The price is work: 41 x 41 x 30 nodes and
20 picks are 1 million travel times, and the answer is only as fine as the grid.

At a node the three coordinates are fixed, but the origin time is not. You could add a fourth grid axis for
$t_0$; think before you do, because the misfit depends on $t_0$ in a very simple way. Write the residuals at the
node with $t_0 = 0$ as $\rho_i = t_i - T_i$; then $r_i = \rho_i - t_0$ for every pick.

Two habits make a grid search fast in Python: compute the epicentral distances of all nodes to all stations at
once with numpy arrays (M01, glossary `vectorization`), and call `tt` once per depth and phase with all those
distances, not once per node.

### M31-T1 · A grid search over the hypocenter · core · about 75 min

**Goal:** write `grid_search(obs, tt, xs, ys, zs)`: the RMS residual at every node of a grid of trial
hypocenters, each with its best origin time, and the node with the smallest RMS.

**Given:** Part 2 and numpy. Not allowed in this function: `scipy` (it has ready-made grid minimizers), `sklearn`,
`statsmodels`, `torch`, `jax`, `lmfit`, `pyrocko` and the course package.

**Contract:**
- Inputs: `obs`, a table with at least the columns `phase` (`"P"` or `"S"`), `x_km` and `y_km` (the station,
  km east and north of the reference point) and `t_s` (the arrival time, s after a reference moment), with any
  index and in any order (do not change it); `tt`, a function `tt(distance_km, depth_km, phase)` that returns the
  travel times (s) for a numpy array of epicentral distances (km) from a source at one depth (km); `xs`, `ys`,
  `zs`, 1-D arrays of trial positions east, north and down (km).
- At every node `(xs[i], ys[j], zs[k])`: the epicentral distance from the node to each pick's station, the
  predicted travel times from `tt`, the origin time `t0` that makes the RMS smallest, and the RMS itself (all
  picks, P and S together, as in Part 1).
- Output: a dict with `"rms"` (numpy array of shape `(len(xs), len(ys), len(zs))`, `rms[i, j, k]` at node
  `(xs[i], ys[j], zs[k])`), `"t0"` (the same shape: the origin time chosen at each node, s after the same
  reference moment), and `"x_km"`, `"y_km"`, `"z_km"`, `"t0_s"`, `"rms_s"` (floats: the node with the smallest
  RMS, its origin time and its RMS).
- Example: at a node where every predicted travel time is exactly 1.5 s shorter than the observed arrival time,
  the origin time is 1.5 s and the RMS is 0.

**Done when** `st.check("M31-T1", func=grid_search)` passes: on 6 made-up networks (a half-space, and a crust
whose speed grows with depth). With exact picks from a node, your best node must be that node, at RMS 0; with
noisy picks, your RMS and origin time at the true node must be what the noise was built to give, and at 15 random
nodes they must agree with a numerical minimization over the origin time.

**Stuck?** `st.hint("M31-T1", 1)`

In [ ]:
# Before you code: the practice picks carry random errors of 0.05 s. What RMS do you expect at the true hypocenter,
# and will the best node of a 1-km grid be the node nearest to the true hypocenter? Along which direction (east,
# north or depth) do you expect the misfit to change most slowly?
st.predict("M31-T1", """

""")

In [ ]:
def grid_search(obs, tt, xs, ys, zs):
    """RMS residual at every node of a grid (best origin time at each node), and the best node.

    obs        : table with phase ("P"/"S"), x_km, y_km (station position, km) and t_s (arrival time, s)
    tt         : function tt(distance_km, depth_km, phase) -> travel times (s) for an array of distances
    xs, ys, zs : 1-D arrays of trial positions east, north and down (km)
    returns    : {"rms": array (nx, ny, nz), "t0": array (nx, ny, nz), "x_km", "y_km", "z_km", "t0_s", "rms_s"}
    """
    # Step 1: plain arrays from the table: station east and north, phase, arrival time.
    # Step 2: the epicentral distance from every (x, y) node to every pick's station, in one array.
    # Step 3: for each depth and phase, one call of tt with all those distances: the predicted travel times.
    # Step 4: at each node, the residuals for t0 = 0 are rho = t - T. Choose the t0 that makes
    #         sum (rho - t0)^2 smallest (where is the derivative with respect to t0 zero?), then the RMS.
    # Step 5: the node with the smallest RMS of the whole cube.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T1: write your code here")

In [ ]:
st.check("M31-T1", func=grid_search)

Your grid search on the practice earthquake, and the RMS on a map at the best depth and on an east-west section
through the best node:

In [ ]:
PRACTICE_X = np.arange(-15.0, 15.01, 1.0)
PRACTICE_Y = np.arange(-15.0, 15.01, 1.0)
PRACTICE_Z = np.arange(0.5, 20.01, 1.0)
start_clock = time.perf_counter()
practice_grid = grid_search(practice, tt_half, PRACTICE_X, PRACTICE_Y, PRACTICE_Z)
print(f"{practice_grid['rms'].size} nodes in {1000.0 * (time.perf_counter() - start_clock):.0f} ms")
print(f"best node: x {practice_grid['x_km']:g}, y {practice_grid['y_km']:g}, z {practice_grid['z_km']:g} km, "
      f"origin time {practice_grid['t0_s']:.3f} s, RMS {practice_grid['rms_s']:.3f} s")
print("true hypocenter:", PRACTICE_TRUTH)
i_best = int(np.argmin(np.abs(PRACTICE_X - practice_grid["x_km"])))
j_best = int(np.argmin(np.abs(PRACTICE_Y - practice_grid["y_km"])))
k_best = int(np.argmin(np.abs(PRACTICE_Z - practice_grid["z_km"])))
close = practice_grid["rms"] <= practice_grid["rms_s"] + 0.05      # nodes within 0.05 s of the best RMS
for label, inside, axis in (("east-west", close[:, j_best, k_best], PRACTICE_X),
                            ("north-south", close[i_best, :, k_best], PRACTICE_Y),
                            ("depth", close[i_best, j_best, :], PRACTICE_Z)):
    print(f"through the best node, {label}: RMS within 0.05 s of the best from {axis[inside].min():g} to "
          f"{axis[inside].max():g} km")

fig, (ax_m, ax_d) = plt.subplots(1, 2, figsize=(10.0, 4.2), width_ratios=[1.0, 1.25])
levels = np.linspace(0.0, 0.6, 25)
mapped = ax_m.contourf(PRACTICE_X, PRACTICE_Y, practice_grid["rms"][:, :, k_best].T, levels=levels,
                       cmap="viridis", extend="max")
ax_m.plot(practice_stations["x_km"], practice_stations["y_km"], "^", color="w", mec="k", ms=7)
ax_m.plot(practice_grid["x_km"], practice_grid["y_km"], "o", color="tab:red", mec="w", ms=7, label="Best node")
ax_m.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["y_km"], "*", color="gold", mec="k", ms=13, label="Truth")
ax_m.set_xlim(PRACTICE_X[0], PRACTICE_X[-1])
ax_m.set_ylim(PRACTICE_Y[0], PRACTICE_Y[-1])
ax_m.set_aspect("equal")
ax_m.set_xlabel("East (km)")
ax_m.set_ylabel("North (km)")
ax_m.set_title(f"(a) Map view at {PRACTICE_Z[k_best]:g} km depth", fontsize=10, loc="left")
ax_d.contourf(PRACTICE_X, PRACTICE_Z, practice_grid["rms"][:, j_best, :].T, levels=levels, cmap="viridis",
              extend="max")
ax_d.plot(practice_grid["x_km"], practice_grid["z_km"], "o", color="tab:red", mec="w", ms=7)
ax_d.plot(PRACTICE_TRUTH["x_km"], PRACTICE_TRUTH["z_km"], "*", color="gold", mec="k", ms=13)
ax_d.set_ylim(PRACTICE_Z[-1], PRACTICE_Z[0])
ax_d.set_xlabel("East (km)")
ax_d.set_ylabel("Depth (km)")
ax_d.set_title(f"(b) East-west section at {PRACTICE_Y[j_best]:g} km north", fontsize=10, loc="left")
fig.colorbar(mapped, ax=ax_d, label="RMS residual (s)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_grid_search.png", dpi=150)

**Explain (3-5 sentences):** How did you choose the origin time at a node, and why does a fourth grid axis for
$t_0$ add nothing? Compare the best node and its RMS (printed above) with the true hypocenter and with your
prediction. In which direction does the RMS change least around the best node (numbers and figure), and why
there?

## Part 3 · Geiger's method (about 40 min, then task 2)

A grid search is robust but coarse. **Geiger's method** (glossary `geiger_method`; Geiger 1910) refines a
location by **linearization** (glossary `linearization`): near a trial hypocenter $m = (x, y, z, t_0)$ the
predicted arrival times change almost linearly with a small change $\delta m$,

$$t_0 + T_i(x + \delta x, y + \delta y, z + \delta z) + \delta t_0 \approx t_0 + T_i(x, y, z)
  + \frac{\partial T_i}{\partial x}\delta x + \frac{\partial T_i}{\partial y}\delta y
  + \frac{\partial T_i}{\partial z}\delta z + \delta t_0 ,$$

so the residuals $r$ should be explained by $G\,\delta m \approx r$. $G$ is the **Jacobian** (glossary
`jacobian`): one row per pick, four columns, $\partial T_i/\partial x$, $\partial T_i/\partial y$,
$\partial T_i/\partial z$ (s/km, for the source position) and 1 (for the origin time). Your `tt` gives times
only, so you compute the derivatives by **finite differences** (glossary `finite_difference`): move the source
by a small $h$ in one direction and back, and divide the change of the time by $2h$ (central differences;
$h = 0.01$ km is small enough and far above rounding errors).

With more picks than unknowns, $G\,\delta m = r$ has no exact solution, so we take the **least-squares**
(glossary `least_squares`) one, which solves the **normal equations** (glossary `normal_equations`)
$G^{\mathsf T} G\,\delta m = G^{\mathsf T} r$ (a 4 x 4 system: `np.linalg.solve`). Then update $m$, compute new
residuals and a new $G$, and repeat: each step is one iteration of the Gauss-Newton method.

Three practical details turn this into a locator (Lee & Lahr 1975; Thurber 1985; Klein 2002):

- **Damping** (glossary `damping`; Levenberg 1944, Marquardt 1963): solve
  $(G^{\mathsf T} G + \lambda^2 I)\,\delta m = G^{\mathsf T} r$ instead. Where the data barely constrain a
  direction, $G^{\mathsf T} G$ is nearly singular and the plain step can be huge; $\lambda^2$ keeps it short. At
  the solution, $G^{\mathsf T} r = 0$, so damping changes the path, not where the iteration ends.
- A **stopping rule** (glossary `stopping_rule`): stop when a step moves the hypocenter by less than about a metre
  (converged), or after a fixed number of steps (not converged: report it).
- **Airquakes** (glossary `airquake`): a step can put the source above the surface. A common fix, and the rule
  of this module, is to halve the depth instead of taking that step.

### M31-T2 · Geiger's method · core · about 120 min

**Goal:** write `geiger(obs, tt, start, damping=0.01, max_iter=30, tol_km=0.001)`: iterate damped least-squares
steps from a starting hypocenter and origin time until the hypocenter stops moving.

**Given:** Part 3, numpy (`np.linalg.solve` is allowed). Not allowed in this function: `scipy` (its least-squares
solvers would do the task; the check uses them to compare), `sklearn`, `statsmodels`, `torch`, `jax`,
`tensorflow`, `lmfit`, `sympy`, `mpmath`, `pyrocko` and the course package.

**Contract:**
- Inputs: `obs` and `tt` as in task 1; `start`, a tuple `(x_km, y_km, z_km, t0_s)`; `damping` ($\lambda \ge 0$);
  `max_iter`; `tol_km`.
- One step: the residuals $r$ and the Jacobian $G$ at the current hypocenter (central differences of `tt` with
  $h$ = 0.01 km for the three space columns, a column of ones for the origin time); $\delta m$ solves
  $(G^{\mathsf T} G + \lambda^2 I)\,\delta m = G^{\mathsf T} r$; then $x, y, z, t_0$ each grow by their part of
  $\delta m$.
- Depth rule: if $z + \delta z < 0$, the new depth is $z / 2$ instead (the other three still take their step).
- Stop after a step whose length in space, $\sqrt{\delta x^2 + \delta y^2 + \delta z^2}$, is below `tol_km`
  (converged), or after `max_iter` steps (not converged).
- Output: a dict with `"x_km"`, `"y_km"`, `"z_km"`, `"t0_s"` (floats after the last step), `"rms_s"` (the RMS
  residual there), `"residual_s"` (1-D numpy array, the residuals there, one per row of `obs` in its order),
  `"G"` (numpy array of shape `(len(obs), 4)`, the Jacobian there, columns x, y, z, origin time), `"n_iter"`
  (int, the number of steps taken) and `"converged"` (bool).

**Done when** `st.check("M31-T2", func=geiger)` passes: on made-up networks in a half-space and in a crust whose
speed grows with depth. Exact picks: you must reach the true hypocenter (within 5 m and 0.5 ms), even from a start
12 km away; noisy picks: you must end where SciPy's `least_squares` ends on the same residuals; one step with
damping must equal the step of the normal equations; a source 0.6 km deep must never leave the ground; and your
Jacobian must match the exact derivatives of the half-space.

**Stuck?** `st.hint("M31-T2", 1)`

In [ ]:
# Before you code: how many steps will Geiger's method need from the best node of your grid search on the practice
# earthquake, and how many from a start about 19 km away, at the surface or 5 km deep? Will the final hypocenter be
# closer to the truth than the best node? What would happen if you left the origin time out of the unknowns and
# kept the one of the start?
st.predict("M31-T2", """

""")

In [ ]:
def geiger(obs, tt, start, damping=0.01, max_iter=30, tol_km=0.001):
    """Geiger's method: damped least-squares steps from start until the hypocenter stops moving.

    obs      : table with phase, x_km, y_km, t_s (as in task 1)
    tt       : function tt(distance_km, depth_km, phase) -> travel times (s)
    start    : (x_km, y_km, z_km, t0_s) to start from
    damping  : lambda in (G^T G + lambda^2 I) dm = G^T r
    max_iter : largest number of steps
    tol_km   : stop when a step moves the hypocenter by less than this (km)
    returns  : {"x_km", "y_km", "z_km", "t0_s", "rms_s", "residual_s", "G", "n_iter", "converged"}
    """
    # Step 1: plain arrays from the table, and a small function that returns the predicted travel time of every
    #         pick for a trial (x, y, z) (as in task 1, for one point).
    # Step 2: the Jacobian: column 1 = (T(x + h) - T(x - h)) / (2 h), the same for y and z, column 4 = ones.
    # Step 3: one step: r = t - t0 - T; solve (G^T G + damping^2 I) dm = G^T r; update; apply the depth rule.
    # Step 4: repeat until the step in space is shorter than tol_km or max_iter steps are done; then the
    #         residuals, the RMS and the Jacobian at the final hypocenter.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T2: write your code here")

In [ ]:
st.check("M31-T2", func=geiger)

Your `geiger` on the practice earthquake from three starts: the best node of your grid search, and two poor starts
about 19 km away, one at the surface and one 5 km deep. The tables follow the poor starts one step at a time (your
function called with `max_iter` = 1, 2, 3, ...); the last column is the largest $|\partial T_i/\partial z|$ of the
Jacobian `G` your function returns after those steps.

In [ ]:
from_grid = geiger(practice, tt_half, (practice_grid["x_km"], practice_grid["y_km"], practice_grid["z_km"],
                                       practice_grid["t0_s"]))
print(f"from the grid's best node: {from_grid['n_iter']} steps, converged {from_grid['converged']}: x "
      f"{from_grid['x_km']:.3f}, y {from_grid['y_km']:.3f}, z {from_grid['z_km']:.3f} km, origin time "
      f"{from_grid['t0_s']:.3f} s, RMS {from_grid['rms_s']:.4f} s")
miss = np.sqrt((from_grid["x_km"] - PRACTICE_TRUTH["x_km"]) ** 2 + (from_grid["y_km"] - PRACTICE_TRUTH["y_km"]) ** 2
               + (from_grid["z_km"] - PRACTICE_TRUTH["z_km"]) ** 2)
print(f"distance from the true hypocenter: {miss:.3f} km")
for label, poor_start in (("start at the surface", (-12.0, 9.0, 0.0, 0.0)),
                          ("start 5 km deep", (-12.0, 9.0, 5.0, 0.0))):
    path = []
    for k in range(1, 11):
        fit = geiger(practice, tt_half, poor_start, max_iter=k)
        path.append({"steps": fit["n_iter"], "x_km": fit["x_km"], "y_km": fit["y_km"], "z_km": fit["z_km"],
                     "t0_s": fit["t0_s"], "rms_s": fit["rms_s"], "converged": fit["converged"],
                     "max |dT/dz| (s/km)": float(np.abs(np.asarray(fit["G"])[:, 2]).max())})
        if fit["converged"]:
            break
    print(f"\n{label} {poor_start}:")
    print(pd.DataFrame(path).round(4).to_string(index=False))

**Explain (3-5 sentences):** How many steps did each start need (printed above), and where did each one end?
How far is your final hypocenter from the true one, and why can it not be exact? From the surface start, why does
the depth never leave 0 km (read the last column of its table, and work out what the normal equations then give for
the depth step; did the depth rule of the Contract act)? Your function reports that start as converged: what does
a stopping rule tell you, and what does it not? Why do you need both methods, a grid search and Geiger's method, in
practice?

## Part 4 · How well is the location known? (about 30 min, then task 3)

Picks have errors, so a location has errors too. If every pick error is random, independent and has the same
standard deviation $\sigma$, and the problem is close to linear near the solution, then the four unknowns have
the **covariance matrix** (glossary `covariance_matrix`)

$$C = \sigma^2 \left(G^{\mathsf T} G\right)^{-1},$$

with $G$ the Jacobian at the solution. Its diagonal holds the variances; their square roots are the **standard
errors** (glossary `standard_error`) of $x, y, z$ (km) and $t_0$ (s): how much each would scatter if the
earthquake could be recorded again with new pick errors. The off-diagonal terms say how the errors go together.
Divided by the two standard errors they give the **error correlation** (glossary `error_correlation`), a
correlation coefficient from -1 to 1: near ±1, a change of one unknown can be compensated by a change of the
other, a **parameter trade-off** (glossary `parameter_trade_off`) that the data cannot tell apart. The upper-left
2 x 2 block of $C$ (east and north) gives the **error ellipse** (glossary `error_ellipse`) of the epicenter:
`np.linalg.eigh` of that block returns two numbers and two directions (its eigenvalues and eigenvectors); the
ellipse's axes point along those directions, and their half-lengths are the square roots of the two numbers.

Where does $\sigma$ come from? From the picker's own estimate, or from the residuals of the location itself:
with $N$ picks and four unknowns, $\hat\sigma = \mathrm{RMS}\sqrt{N/(N-4)}$ (the four unknowns used up four
degrees of freedom). Real residuals also hold the error of the velocity model, which is not random from pick to
pick; formal errors are therefore a lower limit of the true error (Husen & Hardebeck 2010).

The check of the next task uses a **Monte Carlo simulation** (glossary `monte_carlo_simulation`) with your own
`geiger`: it relocates a made-up earthquake 300 times, each time with new random pick errors of a known size, and
compares the scatter of the 300 locations with your standard errors.

### M31-T3 · Errors of a location · core · about 60 min

**Goal:** write `location_errors(G, sigma_s)`: the covariance matrix of a location, its four standard errors and
the correlation coefficient of depth and origin time.

**Given:** Part 4 and numpy (`np.linalg` is allowed). Not allowed: `scipy` (its curve fitting returns a covariance
matrix; the check uses it to compare), `statsmodels`, `sklearn`, `lmfit`, `numpy.polyfit` and `numpy.ma.polyfit`
(both return one with `cov=True`), `numpy.polynomial`, `xarray`, `torch`, `jax` and the course package.

**Contract:**
- Inputs: `G`, a numpy array of shape `(N, 4)` (as your `geiger` returns it: columns x, y, z in s/km and the
  origin time); `sigma_s`, the standard deviation of one pick error (s), the same for every pick.
- Output: a dict with `"cov"` (4 x 4 numpy array, $\sigma^2 (G^{\mathsf T} G)^{-1}$, in km², km s and s²),
  `"se"` (1-D numpy array of the four standard errors: x, y, z in km, origin time in s) and `"corr_zt"` (float,
  the correlation coefficient of the depth and origin-time errors).

**Done when** `st.check("M31-T3", func=location_errors, locate=geiger)` passes: on 8 made-up Jacobians against
SciPy's `curve_fit` on the same linear problem, then with your `geiger`: 300 relocations of a made-up earthquake
with random pick errors of a known size, whose scatter must match your standard errors within 20 % and your
correlation within 0.15.

**Stuck?** `st.hint("M31-T3", 1)`

In [ ]:
# Before you code: for the practice earthquake, which of x, y and z will have the largest standard error, and
# about how large for pick errors of 0.05 s? Will the depth and origin-time errors be correlated positively or
# negatively, and strongly or weakly?
st.predict("M31-T3", """

""")

In [ ]:
def location_errors(G, sigma_s):
    """Covariance matrix, standard errors and depth-origin time correlation of a location.

    G       : Jacobian at the location, shape (N, 4): columns x, y, z (s/km) and origin time (1)
    sigma_s : standard deviation of one pick error (s)
    returns : {"cov": 4 x 4 array, "se": array of 4 standard errors (x, y, z km; t0 s), "corr_zt": float}
    """
    # Step 1: C = sigma^2 (G^T G)^-1.
    # Step 2: the standard errors are the square roots of the diagonal of C.
    # Step 3: the correlation of depth (index 2) and origin time (index 3): C[2, 3] / (se_z se_t0).
    # YOUR CODE HERE
    raise NotImplementedError("M31-T3: write your code here")

In [ ]:
st.check("M31-T3", func=location_errors, locate=geiger)

Your errors of the practice location, and the same Monte Carlo test the check made, on the practice network: 200
relocations with your `geiger`, each from new random pick errors of 0.05 s added to exact times from your
location.

In [ ]:
practice_err = location_errors(from_grid["G"], 0.05)
print("your standard errors for 0.05 s pick errors: x {:.3f}, y {:.3f}, z {:.3f} km, origin time {:.3f} s; "
      "depth-origin time correlation {:.2f}".format(*practice_err["se"], practice_err["corr_zt"]))
mc_rng = np.random.default_rng(PRACTICE_SEED + 1)
exact = practice.copy()
exact["t_s"] = practice["t_s"].to_numpy() - from_grid["residual_s"]        # times that fit your location exactly
cloud = []
for _ in range(200):
    noisy = exact.copy()
    noisy["t_s"] = exact["t_s"].to_numpy() + mc_rng.normal(0.0, 0.05, len(exact))
    fit = geiger(noisy, tt_half, (from_grid["x_km"], from_grid["y_km"], from_grid["z_km"], from_grid["t0_s"]))
    cloud.append([fit["x_km"], fit["y_km"], fit["z_km"], fit["t0_s"]])
cloud = np.array(cloud)
print("scatter of 200 relocations:      x {:.3f}, y {:.3f}, z {:.3f} km, origin time {:.3f} s; "
      "correlation {:.2f}".format(*cloud.std(axis=0), np.corrcoef(cloud[:, 2], cloud[:, 3])[0, 1]))

values, vectors = np.linalg.eigh(practice_err["cov"][:2, :2])
angle = np.linspace(0.0, 2.0 * np.pi, 200)
ellipse = vectors @ (np.sqrt(values)[:, None] * np.vstack([np.cos(angle), np.sin(angle)]))
fig, (ax_e, ax_t) = plt.subplots(1, 2, figsize=(9.6, 4.2))
ax_e.plot(cloud[:, 0], cloud[:, 1], ".", color="0.5", ms=3, label="200 relocations")
ax_e.plot(from_grid["x_km"] + ellipse[0], from_grid["y_km"] + ellipse[1], color="tab:red", lw=1.4,
          label="Your 1-sigma error ellipse")
ax_e.plot(from_grid["x_km"] + 2.0 * ellipse[0], from_grid["y_km"] + 2.0 * ellipse[1], color="tab:red", lw=0.9,
          ls="--", label="2-sigma")
ax_e.set_aspect("equal")
ax_e.set_xlabel("East (km)")
ax_e.set_ylabel("North (km)")
ax_e.set_title("(a) Epicenters", fontsize=10, loc="left")
ax_t.plot(cloud[:, 3], cloud[:, 2], ".", color="0.5", ms=3)
ax_t.set_ylim(ax_t.get_ylim()[::-1])
ax_t.set_xlabel("Origin time (s)")
ax_t.set_ylabel("Depth (km)")
ax_t.set_title("(b) Depth against origin time", fontsize=10, loc="left")
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "practice_monte_carlo.png", dpi=150)

**Explain (3-5 sentences):** Compare your standard errors with the scatter of the 200 relocations (printed
above) and with your prediction. Which unknown is known worst, and why (think of what the stations see)? What do
the sign and size of the depth-origin time correlation mean, and how does panel (b) show it? Name two reasons why
formal errors from real picks are smaller than the true errors of a location.

## Part 5 · The Sangju mainshock from your picks (about 40 min, then task 4)

Now the real earthquake: the ML 3.9 Sangju mainshock, with your manual picks of M20 and the velocity model
`kim1983`. Your two functions of module M30 compute its travel times. Paste `direct_time` and `head_wave` into
the next cell as they are in your M30 notebook (they need numpy only), and run it. If you also paste your
`wadati_fit` of M30, the explorer E5 below draws its line.

In [ ]:
# Paste your direct_time and head_wave (and, if you like, wadati_fit) of module M30 below this line.

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if not callable(globals().get(name))]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above "
                    "and run it (finish task M30-T2 of module M30 first; task M30-T1 writes direct_time).")
MODEL = st.data.velocity_model(MODEL_NAME)
print(f"{MODEL_NAME}: {MODEL.attrs['citation']}")
print(MODEL.to_string(index=False))
catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
EV_LAT, EV_LON, KMA_DEPTH_KM = float(event["latitude"]), float(event["longitude"]), float(event["depth_km"])
print(f"\nKMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), epicenter {EV_LAT:.2f} N {EV_LON:.2f} E, depth "
      f"{KMA_DEPTH_KM:g} km, ML {event['ml']:.1f}")
print(f"source of the catalog: {catalog.attrs.get('source', 'KMA')}")

**Which wave does the locator predict?** A picker marks the **first arrival** (glossary `first_arrival`), so the
locator must predict the first arrival too. In M30 you saw that in kim1983 a source just above the 15-km
boundary sends a **head wave** (glossary `head_wave`) along that boundary, inside the crust, that overtakes the
direct wave at a **crossover distance** (glossary `crossover_distance`) of a few tens of kilometres, and that the
crossover moves inward as the source approaches the boundary. The table below gives your crossover distances for
sources near the KMA depth. So the travel-time function of this module is the earliest of your direct wave and
your head waves along the boundaries inside the crust. The head waves along the Moho, **Pn and Sn** (glossary
`pn_sn`), are left out: the lab never locates with them (their times depend on the poorly known Moho), and within
`MAX_DISTANCE_KM` (100 km) they are not first for these sources anyway (the table shows where they would be).

In [ ]:
cross = []
for depth in (KMA_DEPTH_KM - 4.0, KMA_DEPTH_KM, KMA_DEPTH_KM + 0.9):
    row = {"source depth (km)": depth}
    for n in range(1, len(MODEL)):
        for phase in ("P", "S"):
            label = f"{phase} along {MODEL['top_km'].iloc[n]:g} km" + (" (Moho)" if n == len(MODEL) - 1 else "")
            row[f"{label}: crossover (km)"] = head_wave(np.array([50.0]), depth, MODEL, phase, n)[2]
    cross.append(row)
print("crossover distances of your head_wave (NaN: no head wave from this depth):")
print(pd.DataFrame(cross).round(1).to_string(index=False))

**A travel-time table.** A grid search asks for millions of travel times, and every one of yours is found by
shooting. Real locators compute the times once on a grid of distances and depths, a **travel-time table**
(glossary `travel_time_table`), and interpolate between the nodes later. The helper below does that with your two
functions: every 0.25 km of distance out to 120 km and every 0.1 km of depth, then smooth (cubic spline)
interpolation. The table's depths lie halfway between multiples of 0.1 km, so no node falls exactly on a
boundary of the model (a source on a boundary is a special case of your `direct_time`). It takes a few seconds.

In [ ]:
def first_arrival_tt(model, direct, head, x_max_km=120.0, dx_km=0.25, z_max_km=35.0, dz_km=0.1):
    """tt(distance_km, depth_km, phase) for the locators: the first arrival in `model` from your M30 functions.

    At every node of a table (distances 0..x_max_km, depths dz_km/2, 3 dz_km/2, ... below z_max_km) the time is
    the earliest of direct(...) and head(..., interface=n)[0] for the boundaries n = 1 .. len(model) - 2 (the Moho,
    the last boundary, is left out: no Pn, no Sn). Between the nodes a cubic spline interpolates; outside the
    table the distance and depth are moved to its edge.
    """
    grid_x = np.arange(0.0, x_max_km + 0.5 * dx_km, dx_km)
    grid_z = np.arange(0.5 * dz_km, z_max_km, dz_km)
    splines = {}
    for phase in ("P", "S"):
        table = np.empty((grid_x.size, grid_z.size))
        for col, depth in enumerate(grid_z):
            earliest = np.asarray(direct(grid_x.copy(), float(depth), model, phase), dtype=float)
            for n in range(1, len(model) - 1):
                crustal = np.asarray(head(grid_x.copy(), float(depth), model, phase, n)[0], dtype=float)
                earliest = np.fmin(earliest, crustal)
            table[:, col] = earliest
        splines[phase] = RectBivariateSpline(grid_x, grid_z, table)

    def tt(distance_km, depth_km, phase):
        where = np.clip(np.asarray(distance_km, dtype=float), grid_x[0], grid_x[-1])
        level = np.clip(np.full(where.shape, float(depth_km)), grid_z[0], grid_z[-1])
        return splines[phase].ev(where, level)

    return tt


start_clock = time.perf_counter()
tt_main = first_arrival_tt(MODEL, direct_time, head_wave)
print(f"travel-time table of {MODEL_NAME} built from your functions in {time.perf_counter() - start_clock:.1f} s")
probe = np.array([5.0, 30.0, 60.0, 90.0])
print("check against your direct_time at", probe.tolist(), "km, source at 10 km (P):",
      np.round(tt_main(probe, 10.0, "P") - direct_time(probe, 10.0, MODEL, "P"), 4).tolist(), "s")

**Your picks in the local frame.** The frame is centered on the KMA epicenter. Station positions come from the
SAC headers of your download; times count from the KMA origin time. The cell keeps the first P and the first S
of each station within `MAX_DISTANCE_KM`. If it reports a missing file, finish the picking task of M20 first.

In [ ]:
FRAME = Proj(proj="aeqd", lat_0=EV_LAT, lon_0=EV_LON, datum="WGS84")


def to_local(lat, lon):
    """(x_km, y_km) east and north of the KMA epicenter (azimuthal equidistant projection)."""
    east_m, north_m = FRAME(np.asarray(lon, dtype=float), np.asarray(lat, dtype=float))
    return east_m / 1000.0, north_m / 1000.0


def to_geographic(x_km, y_km):
    """(latitude, longitude) in degrees of a point x_km east and y_km north of the KMA epicenter."""
    lon, lat = FRAME(np.asarray(x_km, dtype=float) * 1000.0, np.asarray(y_km, dtype=float) * 1000.0, inverse=True)
    return float(lat), float(lon)


def station_coordinates(sac_dir, names):
    """(latitude, longitude) of each NET.STA in names, from the SAC headers of one event folder (any band)."""
    found = {}
    for path in sorted(Path(sac_dir).glob("*/*.sac")):
        key = ".".join(path.name.split(".")[1:3])
        if key in names and key not in found:
            head = read(str(path), headonly=True)[0].stats.sac
            found[key] = (float(head.stla), float(head.stlo))
    return found


def observation_table(pick_file, sac_dir, origin, max_km):
    """The table the locators read: station, phase, channel, x_km, y_km, t_s (s after origin), time (UTC).

    One row for the first P and one for the first S of each station whose position is in the SAC headers and that
    lies within max_km of the KMA epicenter.
    """
    picks = pd.read_csv(pick_file, dtype=str)
    picks["time"] = [UTCDateTime(t) for t in picks["time_utc"]]
    picks = picks.sort_values("time").drop_duplicates(["station", "phase"])
    where = station_coordinates(sac_dir, set(picks["station"]))
    rows = []
    for _, pick in picks.iterrows():
        if pick["station"] in where and pick["phase"] in ("P", "S"):
            east, north = to_local(*where[pick["station"]])
            if np.hypot(east, north) <= max_km:
                rows.append({"station": pick["station"], "phase": pick["phase"], "channel": pick["channel"],
                             "x_km": float(east), "y_km": float(north), "t_s": pick["time"] - origin,
                             "time": pick["time"]})
    table = pd.DataFrame(rows)
    table["epi_km"] = np.hypot(table["x_km"], table["y_km"])
    return table.sort_values(["epi_km", "phase"]).reset_index(drop=True)


SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN_KMA.strftime("%Y%m%d%H%M%S")
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. This module locates the mainshock with your own picks: "
                            "finish task M20-T5 of module M20 first (its picker saves the file).")
obs_main = observation_table(PICKS_CSV, SAC_DIR, ORIGIN_KMA, MAX_DISTANCE_KM)
print(f"{len(obs_main)} picks ({(obs_main['phase'] == 'P').sum()} P, {(obs_main['phase'] == 'S').sum()} S) at "
      f"{obs_main['station'].nunique()} stations within {MAX_DISTANCE_KM:g} km of the KMA epicenter")
print(obs_main.drop(columns=["time"]).round(3).to_string(index=False))
GRID_X = np.arange(-GRID_HALF_KM, GRID_HALF_KM + 0.5 * GRID_STEP_KM, GRID_STEP_KM)
GRID_Y = GRID_X.copy()
GRID_Z = np.arange(GRID_DEPTHS_KM[0], GRID_DEPTHS_KM[1] + 0.5 * GRID_DEPTHS_KM[2], GRID_DEPTHS_KM[2])

### M31-T4 · The Sangju mainshock and the KMA catalog · core · about 60 min

**Goal:** locate the Sangju mainshock with your picks, your travel-time table and your three functions, and put
the result in a dictionary `sangju`.

**Given:** `obs_main` and `tt_main` (above), the grid `GRID_X`, `GRID_Y`, `GRID_Z`, and the helper
`to_geographic`.

**Contract (the answer):** first a grid search over `GRID_X`, `GRID_Y`, `GRID_Z`; then `geiger` from its best
node; then `location_errors` with $\hat\sigma = \mathrm{RMS}\sqrt{N/(N-4)}$ from your own residuals (Part 4). Keep
what `geiger` returned as `fit_main` and what `location_errors` returned as `errors_main` (the cells below use
them). Then
`sangju = {"x_km": ..., "y_km": ..., "depth_km": ..., "t0_s": ..., "latitude": ..., "longitude": ...,
"origin_utc": ..., "rms_s": ..., "se": [...], "n_picks": ...}`, with your hypocenter in the local frame (km),
your origin time as seconds after the KMA origin time (`t0_s`) and as ISO 8601 text in UTC (`origin_utc`, for
example `"2031-04-05T06:07:08.090000Z"`, an invented time), your epicenter in degrees, your RMS residual (s),
your four standard errors (x, y, z in km, origin time in s) and the number of picks you used.

**Done when** `st.check("M31-T4", answer=sangju, obs=obs_main, tt=tt_main)` passes. The check recomputes your
RMS from your picks and your travel-time table, tests that no hypocenter 0.25 km away fits better, that your
latitude, longitude and UTC time match your local values, and that your location lies within loose limits of the
KMA catalog (in `criteria/M31.yaml`). It never prints a value.

**Stuck?** `st.hint("M31-T4", 1)`

In [ ]:
# Before you code: KMA gives this epicenter to 0.01 degree (about 1 km) and the depth to the kilometre. How far
# from the KMA hypocenter do you expect your location, horizontally and in depth? Will your origin time be earlier
# or later than KMA's (remember your Wadati origin time of M30)?
st.predict("M31-T4", """

""")

In [ ]:
# Locate the mainshock: grid search, then geiger from its best node, then location_errors with sigma from your
# residuals. Keep fit_main and errors_main, then fill in sangju.
# YOUR CODE HERE
raise NotImplementedError("M31-T4: write your code here")

In [ ]:
st.check("M31-T4", answer=sangju, obs=obs_main, tt=tt_main)

Your location next to the KMA catalog, and your residuals station by station:

In [ ]:
sigma_used = fit_main["rms_s"] * np.sqrt(len(obs_main) / (len(obs_main) - 4))
print(f"your location: {sangju['latitude']:.4f} N {sangju['longitude']:.4f} E, depth {sangju['depth_km']:.2f} "
      f"km, origin {sangju['origin_utc']} ({fit_main['n_iter']} steps, converged {fit_main['converged']})")
print(f"RMS {fit_main['rms_s']:.3f} s from {len(obs_main)} picks; sigma {sigma_used:.3f} s; standard errors x "
      "{:.2f}, y {:.2f}, z {:.2f} km, origin time {:.3f} s; depth-origin time correlation {:.2f}".format(
          *errors_main["se"], errors_main["corr_zt"]))
print(f"KMA catalog:   {EV_LAT:.4f} N {EV_LON:.4f} E, depth {KMA_DEPTH_KM:.2f} km, origin {ORIGIN_KMA}")
print(f"you minus KMA: epicenter {np.hypot(sangju['x_km'], sangju['y_km']):.2f} km away (east "
      f"{sangju['x_km']:+.2f}, north {sangju['y_km']:+.2f} km), depth {sangju['depth_km'] - KMA_DEPTH_KM:+.2f} km, "
      f"origin time {sangju['t0_s']:+.2f} s")
angles_deg = np.degrees(np.arctan2(obs_main["x_km"] - sangju["x_km"], obs_main["y_km"] - sangju["y_km"]))
azimuths = np.sort(np.mod(angles_deg, 360.0))
print(f"largest azimuthal gap of your stations: {np.max(np.diff(np.append(azimuths, azimuths[0] + 360.0))):.0f} "
      "degrees (glossary azimuthal_gap)")
resid_main = obs_main[["station", "phase", "epi_km"]].copy()
resid_main["residual_s"] = fit_main["residual_s"]
for phase in ("P", "S"):
    part = resid_main[resid_main["phase"] == phase]["residual_s"]
    print(f"{phase}: {len(part)} residuals, mean {part.mean():+.3f} s, smallest {part.min():+.3f}, largest "
          f"{part.max():+.3f} s")
print(resid_main.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
    part = resid_main[resid_main["phase"] == phase]
    ax.plot(part["epi_km"], part["residual_s"], "o", color=color, ms=6, label=f"{phase} residuals")
ax.axhline(0.0, color="0.5", lw=0.8)
ax.set_xlabel("Epicentral distance from the KMA epicenter (km)")
ax.set_ylabel("Observed - predicted (s)")
ax.set_title(f"Your residuals at your location, {MODEL_NAME}", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "mainshock_residuals.png", dpi=150)

**Explain (3-5 sentences):** How far is your hypocenter from KMA's, horizontally, in depth and in origin time
(printed above), and are these differences larger than your standard errors and the rounding of the catalog?
What pattern do your P and S residuals show, and which property of kim1983 does it point to (compare M30)? Why
can your RMS be several times the size of your picking uncertainties, and what does that do to the meaning of
your standard errors?

**Explorer E5 (optional, not checked).** The **widget** (glossary `widget`) below draws the RMS of your
`grid_search` around the mainshock in map view at a depth you choose and in a depth section through a line you
choose, and a small **Wadati diagram** (glossary `wadati_diagram`) of the same picks. It computes no misfit
itself: every RMS comes from your function. Things to try: leave out the nearest station, then the two nearest,
and watch the depth section; leave out all S picks of the far stations; move the section line through the
epicenter and away from it.

In [ ]:
# Optional explorer E5 (not checked): misfit slices of your grid_search; leave stations out and watch.
st.widgets.misfit_explorer(obs_main, tt_main, grid_search=grid_search, wadati=globals().get("wadati_fit"),
                           marks={"KMA hypocenter": (0.0, 0.0, KMA_DEPTH_KM)}, half_km=GRID_HALF_KM,
                           step_km=GRID_STEP_KM, depths=GRID_Z, depth_km=sangju["depth_km"])

## Part 6 · A small earthquake in 2018 (about 40 min of picking, then task 5)

Seventeen months before the mainshock, an ML 1.4 earthquake occurred in the same area. The KMA catalog lists it
with a depth (printed below). A small earthquake is recorded well only by the stations close to it, so its
location rests on few picks. In this part you pick it, locate it with your functions, and look at what the
station nearest to it contributes.

**Pick it.** Use the picker E2 of M20 on the `SMALL_BAND` records of this earthquake: P on the vertical channel,
S on a horizontal one, with uncertainties, at as many stations within `SMALL_MAX_KM` as you can read, and at
least at the nearest station; press Save. The depth scans after task 5 leave out the nearest and the two nearest
stations and need at least 5 stations in every set, so pick at least 7. The records are weak: a causal band-pass of 2-15 Hz (in the picker)
helps, a zero-phase one would make the picks early (M10). Leave out a station whose onset you cannot see. Picking
takes about 40 minutes.

In [ ]:
small = catalog.loc[catalog["event_id"] == SMALL_EVENT_ID].iloc[0]
ORIGIN_SMALL = UTCDateTime(small["origin_utc"].isoformat())
SMALL_LAT, SMALL_LON = float(small["latitude"]), float(small["longitude"])
print(f"KMA {SMALL_EVENT_ID}: origin {ORIGIN_SMALL} (UTC), epicenter {SMALL_LAT:.2f} N {SMALL_LON:.2f} E, depth "
      f"{float(small['depth_km']):g} km, ML {small['ml']:.1f}")
SAC_DIR_SMALL = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN_SMALL.strftime("%Y%m%d%H%M%S")
if not (SAC_DIR_SMALL / SMALL_BAND).is_dir():
    raise FileNotFoundError(f"There are no {SMALL_BAND} waveforms of this event in {SAC_DIR_SMALL}. Fetch the "
                            f"{DATASET} data with STP first (module M03).")

In [ ]:
# The picker E2: one station at a time; your picks go to SMALL_PICKS_CSV when you press Save.
picker_small = st.widgets.phase_picker(SAC_DIR_SMALL / SMALL_BAND, origin=ORIGIN_SMALL,
                                       epicenter=(SMALL_LAT, SMALL_LON), max_km=SMALL_MAX_KM,
                                       csv_path=SMALL_PICKS_CSV, window=(-2.0, 25.0), fmin=2.0, fmax=15.0)
picker_small

Your 2018 picks in the same local frame (centered on the KMA epicenter of the mainshock) and with the same
travel-time table; times count from the KMA origin time of the 2018 earthquake.

In [ ]:
if not Path(SMALL_PICKS_CSV).is_file():
    raise FileNotFoundError(f"{SMALL_PICKS_CSV} is missing: pick the 2018 earthquake in the picker above and press "
                            "Save (or call picker_small.save()).")
obs_small = observation_table(SMALL_PICKS_CSV, SAC_DIR_SMALL, ORIGIN_SMALL, MAX_DISTANCE_KM)
NEAREST = obs_small.sort_values("epi_km")["station"].drop_duplicates().tolist()
print(f"{len(obs_small)} picks at {len(NEAREST)} stations; nearest first: {NEAREST}")
print(obs_small.drop(columns=["time"]).round(3).to_string(index=False))

### M31-T5 · A depth scan · core · about 75 min

**Goal:** write `depth_scan(obs, tt, depths, start)`: for every depth of a list, the best epicenter and origin
time with the depth held fixed, and the RMS there. The curve of RMS against depth is a **depth scan** (glossary
`depth_scan`).

**Given:** your `geiger` (task 2), whose step you can reuse with three unknowns. The cell below the task holds a
colleague's location of the 2018 earthquake: run it as it is before your prediction. Not allowed in this
function: the same libraries as in task 2.

**Contract:**
- Inputs: `obs` and `tt` as in task 1; `depths`, a 1-D array of depths (km); `start`, a tuple `(x_km, y_km,
  t0_s)` for the first depth.
- At each depth, in the order given: Geiger's method on x, y and $t_0$ only (the columns x, y and the ones of the
  Jacobian; central differences with h = 0.01 km; no damping), with the depth held at that value, starting from
  the answer at the depth before (the first one from `start`); stop when a step moves the epicenter by less than
  0.001 km, or after 30 steps.
- Output: a pandas DataFrame with the columns `depth_km`, `x_km`, `y_km`, `t0_s` and `rms_s`, one row per depth
  in the order given.

**Done when** `st.check("M31-T5", func=depth_scan)` passes: on made-up networks (a half-space and a crust whose
speed grows with depth), exact and noisy picks; at every depth your epicenter, origin time and RMS must agree with
SciPy's `least_squares` on the same three unknowns, and with exact picks the true depth must give RMS 0.

**Stuck?** `st.hint("M31-T5", 1)`

In [ ]:
# A colleague located the 2018 earthquake with your grid_search and geiger twice: with all your picks, and without
# the station nearest to the epicenter. Run it as it is.
colleague = {}
without_nearest = obs_small[obs_small["station"] != NEAREST[0]]
for label, table in (("all stations", obs_small), (f"without {NEAREST[0]}", without_nearest)):
    table = table.reset_index(drop=True)
    first_guess = grid_search(table, tt_main, GRID_X, GRID_Y, GRID_Z)
    colleague[label] = geiger(table, tt_main, (first_guess["x_km"], first_guess["y_km"], first_guess["z_km"],
                                               first_guess["t0_s"]))
    print(f"colleague, {label}: depth {colleague[label]['z_km']:.1f} km, RMS {colleague[label]['rms_s']:.3f} s, "
          f"{len(table)} picks")
print("colleague's note: both fits have about the same RMS, so the nearest station can be left out without loss; "
      "the depth of the 2018 earthquake is known either way.")

In [ ]:
# Before you code: do you agree with the colleague's note? Over which range of depths would the RMS stay close to
# its smallest value, with and without the nearest station? As the fixed depth grows, what will the best origin
# time do, and why?
st.predict("M31-T5", """

""")

In [ ]:
def depth_scan(obs, tt, depths, start):
    """Best epicenter and origin time, and the RMS, at each of a list of fixed depths.

    obs    : table with phase, x_km, y_km, t_s (as in task 1)
    tt     : function tt(distance_km, depth_km, phase) -> travel times (s)
    depths : 1-D array of fixed depths (km)
    start  : (x_km, y_km, t0_s) for the first depth
    returns: pandas DataFrame with depth_km, x_km, y_km, t0_s, rms_s (one row per depth, in the order given)
    """
    # Step 1: the predicted times of a trial (x, y) at a fixed depth (as in task 2).
    # Step 2: at one depth, Geiger's step with three columns (x, y, ones): solve G^T G dm = G^T r, update
    #         x, y, t0, repeat until the epicenter moves less than 0.001 km (at most 30 steps).
    # Step 3: start each depth from the answer of the one before; record the RMS at the end.
    # YOUR CODE HERE
    raise NotImplementedError("M31-T5: write your code here")

In [ ]:
st.check("M31-T5", func=depth_scan)

Your depth scan of the 2018 earthquake with all stations, without the nearest one, and without the two nearest,
and the standard errors of your `location_errors` at each `geiger` solution ($\hat\sigma$ from each fit's own
residuals). A depth whose misfit is within one standard error of the best one is one where
$N(\mathrm{RMS}^2 - \mathrm{RMS}_{\min}^2)/\hat\sigma^2 \le 1$: the sum of squared residuals has grown by no more
than one pick variance. The cell prints that range of depths too.

In [ ]:
SCAN = np.arange(SCAN_DEPTHS_KM[0], SCAN_DEPTHS_KM[1] + 0.5 * SCAN_DEPTHS_KM[2], SCAN_DEPTHS_KM[2])
sets = {"all stations": obs_small,
        f"without {NEAREST[0]}": obs_small[obs_small["station"] != NEAREST[0]],
        f"without {NEAREST[0]} and {NEAREST[1]}": obs_small[~obs_small["station"].isin(NEAREST[:2])]}
scans, fits_small, summary = {}, {}, []
for label, table in sets.items():
    table = table.reset_index(drop=True)
    n_stations = table["station"].nunique()
    if n_stations < 5:                           # 4 unknowns: sigma needs more picks than that, and a scan needs more
        raise ValueError(f"The set '{label}' has picks at {n_stations} stations ({len(table)} picks); every set needs "
                         "at least 5 stations. Pick at least 7 stations of the 2018 earthquake in the picker above "
                         "and press Save.")
    guess = grid_search(table, tt_main, GRID_X, GRID_Y, GRID_Z)
    fit = geiger(table, tt_main, (guess["x_km"], guess["y_km"], guess["z_km"], guess["t0_s"]))
    n = len(table)
    sigma = fit["rms_s"] * np.sqrt(n / (n - 4))
    try:
        err = location_errors(fit["G"], sigma)
        se_depth, corr_zt = err["se"][2], err["corr_zt"]
    except np.linalg.LinAlgError:                # G^T G cannot be inverted: some direction is not fixed at all
        err, se_depth, corr_zt = None, np.inf, np.nan
    scan = depth_scan(table, tt_main, SCAN, (fit["x_km"], fit["y_km"], fit["t0_s"]))
    scans[label], fits_small[label] = scan, (fit, err)
    best = scan["rms_s"].min()
    inside = scan["depth_km"][n * (scan["rms_s"] ** 2 - best ** 2) / sigma ** 2 <= 1.0]
    around = np.abs(scan["depth_km"] - scan.loc[scan["rms_s"].idxmin(), "depth_km"]) <= 3.0
    slope = np.polyfit(scan["depth_km"][around], scan["t0_s"][around], 1)[0]    # within 3 km of the best depth
    summary.append({"picks": label, "N": n, "depth (km)": fit["z_km"], "RMS (s)": fit["rms_s"],
                    "sigma (s)": sigma, "se depth (km)": se_depth, "corr depth-t0": corr_zt,
                    "scan: best depth (km)": scan.loc[scan["rms_s"].idxmin(), "depth_km"],
                    "scan: depths within 1 se (km)": f"{inside.min():g}-{inside.max():g}",
                    "scan: t0 change per km near the best (s)": slope})
print(pd.DataFrame(summary).round(3).to_string(index=False))
for label, scan in scans.items():                # every depth where the RMS is lower than at both neighbours
    r = scan["rms_s"].to_numpy()
    lower_left = np.r_[True, r[1:] < r[:-1]]
    lower_right = np.r_[r[:-1] <= r[1:], True]
    minima = scan["depth_km"][lower_left & lower_right]
    print(f"{label}: local minima of the RMS at", ", ".join(f"{d:g} km" for d in minima))
print(f"KMA catalog depth of this earthquake: {float(small['depth_km']):g} km; your mainshock depth: "
      f"{sangju['depth_km']:.2f} km")

fig, (ax_r, ax_t) = plt.subplots(1, 2, figsize=(10.0, 4.0))
for (label, scan), color in zip(scans.items(), ("k", "tab:orange", "tab:purple")):
    ax_r.plot(scan["depth_km"], scan["rms_s"], color=color, lw=1.5, label=label)
    ax_t.plot(scan["depth_km"], scan["t0_s"], color=color, lw=1.5)
ax_r.axvline(float(small["depth_km"]), color="0.5", ls=":", lw=1.0, label="KMA catalog depth")
ax_t.axvline(float(small["depth_km"]), color="0.5", ls=":", lw=1.0)
ax_r.set_xlabel("Fixed depth (km)")
ax_r.set_ylabel("RMS residual (s)")
ax_r.set_title("(a) Misfit against depth", fontsize=10, loc="left")
ax_t.set_xlabel("Fixed depth (km)")
ax_t.set_ylabel("Best origin time after KMA's (s)")
ax_t.set_title("(b) Origin time against depth", fontsize=10, loc="left")
fig.legend(loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "depth_scan_2018.png", dpi=150)

**Explain (3-5 sentences):** Answer the colleague with your numbers (printed above): the depth, its standard error
and the range of depths within one standard error, with and without the nearest station and the two nearest, and
where each depth scan has its local minima. Why does the origin time change with the fixed depth (panel b), and why
does that make the depth hard to fix without a near station? Is the 2018 earthquake shallower than the mainshock,
and with which picks can you say so? What would you do in practice when the nearest station is missing?

**The two earthquakes on a map** (PyGMT, as in M02): (a) the stations you used, (b) your two epicenters with their
1-sigma error ellipses and the KMA epicenters, in a box of a few kilometres.

In [ ]:
import pygmt

fit_small, err_small = fits_small["all stations"]
lat_small, lon_small = to_geographic(fit_small["x_km"], fit_small["y_km"])
sta_where = station_coordinates(SAC_DIR, set(obs_main["station"]))
sta_where.update(station_coordinates(SAC_DIR_SMALL, set(obs_small["station"])))
sta_lat = np.array([v[0] for v in sta_where.values()])
sta_lon = np.array([v[1] for v in sta_where.values()])


def ellipse_row(lon, lat, cov):
    """[lon, lat, azimuth (deg from north), major, minor (km, full axes)] of the 1-sigma ellipse of cov[:2, :2]."""
    values, vectors = np.linalg.eigh(np.asarray(cov)[:2, :2])
    major = vectors[:, 1]
    azimuth = float(np.degrees(np.arctan2(major[0], major[1])) % 180.0)
    return [lon, lat, azimuth, 2.0 * np.sqrt(values[1]), 2.0 * np.sqrt(values[0])]


region_a = [float(sta_lon.min()) - 0.1, float(sta_lon.max()) + 0.1, float(sta_lat.min()) - 0.08,
            float(sta_lat.max()) + 0.08]
region_b = [EV_LON - 0.06, EV_LON + 0.06, EV_LAT - 0.045, EV_LAT + 0.045]
with st.plot.gmt_style(FONT_TITLE="11p,Helvetica"):
    gmt_fig = pygmt.Figure()
    gmt_fig.coast(region=region_a, projection="M8c", land="gray95", water="white", shorelines="0.25p,gray50")
    gmt_fig.plot(x=sta_lon, y=sta_lat, style="t0.25c", fill="royalblue", pen="0.4p,black", label="Stations used")
    gmt_fig.plot(x=[region_b[0], region_b[1], region_b[1], region_b[0], region_b[0]],
                 y=[region_b[2], region_b[2], region_b[3], region_b[3], region_b[2]], pen="0.8p,red",
                 label="Box of panel b")
    gmt_fig.legend(position="jBL+o0.15c", box="+gwhite+p0.4p")
    gmt_fig.basemap(frame=["af", "+t(a) Stations"], map_scale=st.plot.scale_bar_spec(region_a, corner="BR"))
    gmt_fig.shift_origin(xshift="9.2c")
    gmt_fig.coast(region=region_b, projection="M8c", land="gray95", water="white", shorelines="0.25p,gray50")
    gmt_fig.plot(data=[ellipse_row(sangju["longitude"], sangju["latitude"], errors_main["cov"])], style="E",
                 pen="1p,red3")
    gmt_fig.plot(x=[sangju["longitude"]], y=[sangju["latitude"]], style="a0.35c", fill="red3", pen="0.4p,black",
                 label="Your mainshock 2019")
    gmt_fig.plot(data=[ellipse_row(lon_small, lat_small, err_small["cov"])], style="E", pen="1p,darkorange")
    gmt_fig.plot(x=[lon_small], y=[lat_small], style="a0.3c", fill="darkorange", pen="0.4p,black",
                 label="Your 2018 earthquake")
    gmt_fig.plot(x=[EV_LON], y=[EV_LAT], style="x0.3c", pen="1.5p,red3", label="KMA 2019")
    gmt_fig.plot(x=[SMALL_LON], y=[SMALL_LAT], style="x0.3c", pen="1.5p,darkorange", label="KMA 2018")
    gmt_fig.legend(position="jTL+o0.15c", box="+gwhite+p0.4p")
    gmt_fig.basemap(frame=["af", "+t(b) Epicenters"], map_scale=st.plot.scale_bar_spec(region_b, corner="BR"))
gmt_fig.savefig(Path(FIGURE_DIR) / "sangju_epicenters_map.png", dpi=200)
gmt_fig.show()

## Part 7 · HYPOINVERSE, a production locator (about 20 min, then task 6)

**HYPOINVERSE** (glossary `hypoinverse`; Klein 2002) is the locator of the US Geological Survey and many networks,
and the one PocketQuake runs. It uses Geiger's method in a layered model, with a few production habits your
locator does not have: **distance weighting** (glossary `distance_weighting`), which weakens far stations, and
**residual weighting** (glossary `residual_weighting`), which weakens picks with large residuals after the first
iterations; it starts every earthquake at a **trial depth** (glossary `trial_depth`) set by the command `ZTR`.

It reads plain text files: a station file, one model file per phase (`CRH`: speed and depth of the top of each
layer), a **phase file** (glossary `phase_file`) with the picks of each earthquake, and commands, and it writes a
**summary file** (glossary `summary_file`) with one line per earthquake. The phase file is in a **fixed-width
format** (glossary `fixed_width_format`; the Y2000 format of the manual): every value has its own columns in the
line, numbers carry an **implied decimal point** (glossary `implied_decimal_point`; the seconds 8.25 are written
`825`), and each pick has a **weight code** (glossary `pick_weight_code`) from 0 (full weight) to 4 (none). A value
one column off is read as something else. The helpers below write the station and model files, put a header and a
closing line around your phase lines, and run HYPOINVERSE with the commands of the lab's PocketQuake pipeline:
`HYP_TEMPLATE` in `pipeline/core/hypoinverse.py` of your PocketQuake copy, with the values of `STD_HYP` in
`pipeline/clusters/_base.py` (and `LET` from `HypControl` in `pipeline/config.py`). The cell below also prints
PocketQuake's lines next to the ones sent here: they are the same, line by line, except the file names
(PocketQuake names its files after the region and the model; here every run has its own folder with
`stations.sta`, `model_p.crh`, `model_s.crh`, `event.phs`, `run.prt`, `run.sum` and `run.arc`). Two commands do not
move a location: `FIL` makes HYPOINVERSE look at the phase file and set the formats of the phase and archive
files itself (without it, or commands that set those formats, it reads no earthquake at all), and `REP T T`
reports each earthquake on the screen and lists the readings of zero weight in the print file too. Your task is
the phase lines.

In [ ]:
HYP_COMMANDS = """REP T T
CON 50
MIN 4
ZTR {trial:g} F
DIS 4 50 1 3
RMS 4 0.12 2 4
ERF T
TOP F
LST 2 0 1
KPR 3
H71 4 1 3
LET 5 2 0 0 0
STA 'stations.sta'
CRH 1 'model_p.crh'
CRH 2 'model_s.crh'
SAL 1 2
PHS 'event.phs'
FIL
PRT 'run.prt'
SUM 'run.sum'
ARC 'run.arc'
LOC
STO
"""


def hypoinverse_program():
    """The path of hyp1.40, which bin/sgtl-install tools put in ~/sgtl/tools/bin."""
    search = os.pathsep.join(p for p in (st.pq.tools_env().get("TOOLS_PATH", ""), os.environ.get("PATH", "")) if p)
    found = shutil.which("hyp1.40", path=search)
    if found is None:
        raise FileNotFoundError("HYPOINVERSE (hyp1.40) is not installed. In a terminal: bin/sgtl-install tools "
                                "(read the dry run), then bin/sgtl-install tools --apply")
    return found


def deg_min(value, positive, negative):
    """Whole degrees, decimal minutes and the hemisphere letter of a latitude or longitude."""
    whole = int(abs(value))
    return whole, (abs(value) - whole) * 60.0, positive if value >= 0 else negative


def write_hypoinverse_files(folder, picks, coords, model, origin, lat, lon, phase_lines):
    """Station file, P and S model files and the phase file (header, your lines, closing line) in folder."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    with open(folder / "stations.sta", "w") as out:
        for name in sorted(set(picks["station"])):
            network, code = name.split(".")
            la_d, la_m, ns = deg_min(coords[name][0], "N", "S")
            lo_d, lo_m, ew = deg_min(coords[name][1], "E", "W")
            out.write(f"{code:<5} {network:<2}  HHZ  {la_d:2d} {la_m:7.4f}{ns}{lo_d:3d} {lo_m:7.4f}{ew}   0\n")
    for suffix, column in (("p", "vp"), ("s", "vs")):
        layers = "".join(f" {v:.2f} {top:.2f}\n" for v, top in zip(model[column], model["top_km"]))
        (folder / f"model_{suffix}.crh").write_text(f"{MODEL_NAME} {suffix.upper()}\n" + layers)
    la_d, la_m, ns = deg_min(lat, "N", " ")
    lo_d, lo_m, ew = deg_min(lon, "E", "W")
    header = (origin.strftime("%Y%m%d%H%M%S") + f"{int(round(origin.microsecond / 1e4)):02d}"
              + f"{la_d:2d}{ns}{int(la_m * 100):4d}{lo_d:3d}{ew}{int(lo_m * 100):4d}")
    closing = " " * 66 + "1"
    (folder / "event.phs").write_text("\n".join([header, *phase_lines, closing]) + "\n")


def run_hypoinverse(folder, trial_depth_km):
    """Run HYPOINVERSE in folder with the lab's settings and the given trial depth; return its summary line."""
    done = subprocess.run([hypoinverse_program()], input=HYP_COMMANDS.format(trial=trial_depth_km), cwd=folder,
                          capture_output=True, text=True, timeout=300)
    if done.returncode != 0 or not (Path(folder) / "run.sum").is_file():
        raise RuntimeError(f"HYPOINVERSE failed in {folder}: {done.stdout[-800:]}")
    return read_hypoinverse_summary(Path(folder) / "run.sum")


def read_hypoinverse_summary(path):
    """The first earthquake of a HYPOINVERSE summary file (H71 4 1 3, comma-separated) as a dict."""
    table = pd.read_csv(path, skipinitialspace=True)
    table.columns = [str(c).strip() for c in table.columns]
    line = table.iloc[0]
    minute = str(line[table.columns[0]]).strip().replace("/", "-").replace(" ", "T")
    return {"latitude": float(line["LAT"]), "longitude": float(line["LON"]), "depth_km": float(line["DEPTH"]),
            "origin": UTCDateTime(minute + ":00") + float(line["SEC"]), "rms_s": float(line["RMS"]),
            "erh_km": float(line["ERH"]), "erz_km": float(line["ERZ"]), "phases": int(line["NUM"]),
            "gap_deg": int(line["GAP"])}


print("HYPOINVERSE:", hypoinverse_program())

# PocketQuake's location commands (left, from your copy; the values in braces are printed below them) next to the
# commands the helpers send (right, with the trial depth TRIAL_DEPTH_KM).
PQ_PIPELINE = st.pq.pipeline_dir() / "pipeline"
if (PQ_PIPELINE / "core" / "hypoinverse.py").is_file():
    template = (PQ_PIPELINE / "core" / "hypoinverse.py").read_text(encoding="utf-8")
    lab_lines = template.split('HYP_TEMPLATE = """', 1)[1].split('"""', 1)[0].strip().splitlines()
    sent_lines = HYP_COMMANDS.format(trial=TRIAL_DEPTH_KM).strip().splitlines()
    print(f"\n{'PocketQuake (HYP_TEMPLATE)':<36}this module (HYP_COMMANDS)")
    for lab_line, sent_line in zip(lab_lines, sent_lines):
        print(f"{lab_line:<36}{sent_line}")
    print(f"lines: {len(lab_lines)} and {len(sent_lines)}")
    for source, word, count in ((PQ_PIPELINE / "clusters" / "_base.py", "STD_HYP =", 2),
                                (PQ_PIPELINE / "config.py", "LET: tuple", 1)):
        lines = source.read_text(encoding="utf-8").splitlines()
        at = next(k for k, line in enumerate(lines) if word in line)
        print("\n".join(f"{source.name}:{at + 1 + j}: {lines[at + j].strip()}" for j in range(count)))
else:
    print("PocketQuake is not installed in", st.pq.clone_dir(), "(module M21 installs it), so its template cannot be "
          "shown here.")

### M31-T6 · Phase lines for HYPOINVERSE · core · about 60 min

**Goal:** write `hypoinverse_lines(picks)`: one line of HYPOINVERSE's phase file per pick, then run HYPOINVERSE on
your mainshock picks three times (first run, unchanged rerun, one setting changed).

**Given:** the helpers above and the column table below (from the HYPOINVERSE manual, Klein 2002). Not allowed in
this function: PocketQuake's code (`pipeline`, `pocketquake`, which writes such files), `obspy.io` and the course
package.

**Contract:**
- Input: `picks`, a table with the columns `station` (`NET.STA`: a network code of 2 characters, a station code
  of 1 to 5), `phase` (`"P"` or `"S"`), `channel` (3 characters), `time` (an obspy `UTCDateTime` in UTC) and
  `weight` (an integer 0-4; 0 = full weight), in any order (do not change it).
- Output: a list of strings, one per row of `picks` in its order. Columns are counted from 1:

  | columns | P line | S line |
  |---|---|---|
  | 1-5 | station code, left-justified | the same |
  | 6-7 | network code | the same |
  | 10-12 | channel | the same |
  | 14-15 | `IP` | blank |
  | 17 | weight digit | blank |
  | 18-29 | year, month, day, hour, minute of the pick: `YYYYMMDDHHMM` | the same |
  | 30-34 | seconds after that minute in hundredths, right-justified (8.25 s: `  825`) | blank |
  | 42-46 | blank | seconds after that minute in hundredths, right-justified |
  | 47-48 | blank | `ES` |
  | 50 | blank | weight digit |

  Every other column up to the last one used is blank. Round the time to the nearest 0.01 s first, then take
  its minute and seconds.
- Example (invented): station `XX.AB1`, P on `HHZ` at `2031-04-05T06:07:08.254Z`, weight 0, under a ruler of
  column numbers:

  ```text
           1         2         3         4         5
  12345678901234567890123456789012345678901234567890
  AB1  XX  HHZ IP 0203104050607  825
  ```

**Done when** `st.check("M31-T6", func=hypoinverse_lines)` passes: on made-up picks (invented stations and dates,
times on the edge of a second and of a minute) whose columns the check reads back, then with HYPOINVERSE itself:
your lines must make it locate a made-up earthquake at its true hypocenter.

**Stuck?** `st.hint("M31-T6", 1)`

In [ ]:
# Before you code: with the same picks and the same model, will HYPOINVERSE put the mainshock where your geiger
# did, to within how much? Will a second run write identical files? Will the trial depth CHANGED_TRIAL_DEPTH_KM
# instead of TRIAL_DEPTH_KM change the location, with all stations and without the nearest one?
st.predict("M31-T6", """

""")

In [ ]:
def hypoinverse_lines(picks):
    """HYPOINVERSE phase-file lines (Y2000 format), one per pick, in the order of the table.

    picks   : table with station (NET.STA), phase ("P"/"S"), channel (3 characters), time (UTCDateTime, UTC),
              weight (0-4)
    returns : list of str
    """
    # Step 1: for each pick, split NET.STA; round the time to 0.01 s; its minute as YYYYMMDDHHMM and the seconds
    #         after that minute in hundredths (an integer).
    # Step 2: build the line with f-strings: :<5 pads on the right, :5d right-justifies an integer in 5 columns.
    # Step 3: P and S put the phase, the weight and the seconds in different columns (see the table).
    # YOUR CODE HERE
    raise NotImplementedError("M31-T6: write your code here")

In [ ]:
st.check("M31-T6", func=hypoinverse_lines)

The production runs, on your mainshock picks (all weights 0, as in your own locator): run 1 with the lab's
settings, run 2 the same again, run 3 with one setting changed, the trial depth. Each run has its own folder in
`results/hypoinverse/`. The same change is then tried without the nearest station.

In [ ]:
HYP_DIR = Path(RESULT_DIR) / "hypoinverse"
coords_main = station_coordinates(SAC_DIR, set(obs_main["station"]))
hyp_picks = obs_main[["station", "phase", "channel", "time"]].assign(weight=0)
print("your first three phase lines:", *hypoinverse_lines(hyp_picks)[:3], sep="\n")
runs = {}
for label, trial in (("run 1", TRIAL_DEPTH_KM), ("run 2 (rerun)", TRIAL_DEPTH_KM),
                     (f"run 3 (trial depth {CHANGED_TRIAL_DEPTH_KM:g} km)", CHANGED_TRIAL_DEPTH_KM)):
    folder = HYP_DIR / label.split(" (")[0].replace(" ", "")
    write_hypoinverse_files(folder, hyp_picks, coords_main, MODEL, ORIGIN_KMA, EV_LAT, EV_LON,
                            hypoinverse_lines(hyp_picks))
    runs[label] = run_hypoinverse(folder, trial)
    time.sleep(1.1)                      # one run per second at most, as when you run the cells one by one
for name in ("run.sum", "run.arc", "run.prt"):
    first_lines = (HYP_DIR / "run1" / name).read_text().splitlines()
    rerun_lines = (HYP_DIR / "run2" / name).read_text().splitlines()
    differ = [(a, b) for a, b in zip(first_lines, rerun_lines) if a != b]
    same = filecmp.cmp(HYP_DIR / "run1" / name, HYP_DIR / "run2" / name, shallow=False)
    print(f"run 2 {name} identical to run 1: {same}; lines that differ: {len(differ)}"
          + (f" (run 1: {differ[0][0].strip()!r}; run 2: {differ[0][1].strip()!r})" if differ else ""))

nearest_main = obs_main.sort_values("epi_km")["station"].iloc[0]
fewer = obs_main[obs_main["station"] != nearest_main].reset_index(drop=True)
for trial in (TRIAL_DEPTH_KM, CHANGED_TRIAL_DEPTH_KM):
    folder = HYP_DIR / f"without_nearest_trial{trial:g}"
    picks_fewer = fewer[["station", "phase", "channel", "time"]].assign(weight=0)
    write_hypoinverse_files(folder, picks_fewer, coords_main, MODEL, ORIGIN_KMA, EV_LAT, EV_LON,
                            hypoinverse_lines(picks_fewer))
    runs[f"without {nearest_main}, trial depth {trial:g} km"] = run_hypoinverse(folder, trial)
guess_fewer = grid_search(fewer, tt_main, GRID_X, GRID_Y, GRID_Z)
mine_fewer = geiger(fewer, tt_main, (guess_fewer["x_km"], guess_fewer["y_km"], guess_fewer["z_km"],
                                     guess_fewer["t0_s"]))

lat_fewer, lon_fewer = to_geographic(mine_fewer["x_km"], mine_fewer["y_km"])
located = {"your geiger": {"latitude": sangju["latitude"], "longitude": sangju["longitude"],
                           "depth_km": sangju["depth_km"], "origin": ORIGIN_KMA + sangju["t0_s"],
                           "rms_s": sangju["rms_s"]}}
located.update(list(runs.items())[:3])
located[f"your geiger without {nearest_main}"] = {"latitude": lat_fewer, "longitude": lon_fewer,
                                                  "depth_km": mine_fewer["z_km"],
                                                  "origin": ORIGIN_KMA + mine_fewer["t0_s"],
                                                  "rms_s": mine_fewer["rms_s"]}
located.update(list(runs.items())[3:])
compare = []
for label, loc in located.items():
    east, north = to_local(loc["latitude"], loc["longitude"])
    compare.append({"location": label, "latitude": loc["latitude"], "longitude": loc["longitude"],
                    "east (km)": float(east), "north (km)": float(north), "depth (km)": loc["depth_km"],
                    "origin - KMA (s)": loc["origin"] - ORIGIN_KMA, "RMS (s)": loc["rms_s"],
                    "ERH (km)": loc.get("erh_km", np.nan), "ERZ (km)": loc.get("erz_km", np.nan)})
compare = pd.DataFrame(compare)
print(compare.round(4).to_string(index=False))
compare.to_csv(Path(RESULT_DIR) / "M31_hypoinverse_runs.csv", index=False)

**Explain (3-5 sentences):** Was the rerun identical, file by file, and if one file differs, what in it changes
(open it)? How far is HYPOINVERSE's location from your `geiger` location with the same picks, and what could make
the small difference (weights, rounding, travel times, stopping rules)? What did the changed trial depth do with
all stations and without the nearest one, and how does that agree with the depth scans of Part 6? Compare
HYPOINVERSE's ERH and ERZ with your standard errors.

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M31")

### Going further (optional, not checked)

1. Locate both earthquakes with `kim2011` (set `MODEL_NAME` and rerun from Part 5). Does the P-S split of your
   residuals change, and where does the mainshock depth go now that no boundary lies near it?
2. Weight the picks by your uncertainties: multiply each row of $G$ and $r$ by $1/\sigma_i$ before the normal
   equations. Does the mainshock move?
3. In the explorer E5, leave out all S picks. How do the map view and the depth section change, and why do S
   picks at near stations matter so much for the depth (Gomberg et al. 1990)?
4. Run HYPOINVERSE on the 2018 picks with and without the nearest station and both trial depths. Do its ERZ
   values follow your standard errors of task 5?
5. Read Husen & Hardebeck (2010) on location errors and list three that no formal error describes.

Next: M32 runs HYPOINVERSE inside the lab's PocketQuake workflow on several Sangju earthquakes and reads its
result files, starting from the HYPOINVERSE run of task 6; relative relocation with HypoDD comes in M41.